# Walk and dictation on four algorithms in one shared head

Reproduces the steering numbers from Part 2 of the *Inside the grokked manifold* longform — chord walk, linear tangent, random, and full dictation on a single 4-task transformer that grokked division, addition, max, and parity at P=149, WD=0.3.

**Source model**: seed 1000 final checkpoint, loaded from Hugging Face Hub.

Outputs `walk_results.json` and `dictation_results.json` next to the notebook plus four inline figures.

Runs on a free Colab T4 or local CPU in a few minutes.

In [ ]:
# Cell 1 — setup + load the trained 4-task model from the Hub
import json
from pathlib import Path
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'huggingface_hub'])
    from huggingface_hub import hf_hub_download

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_DIR = Path('.').resolve()

P = 149
OP_DIV, OP_ADD, OP_MAX, OP_PARITY, EQ = P, P+1, P+2, P+3, P+4
VOCAB = P + 5
TASKS = ['div', 'add', 'max', 'parity']
TASK_OP = {'div': OP_DIV, 'add': OP_ADD, 'max': OP_MAX, 'parity': OP_PARITY}
TASK_COLORS = {'div': '#D45A2A', 'add': '#3A6E8C', 'max': '#5E9C5C', 'parity': '#9C5EBE'}

BG, FG, GRID, MUTED = '#FAFAF7', '#2A2A2A', '#E5E3DD', '#6E6E6E'
plt.rcParams.update({
    'figure.facecolor': BG, 'axes.facecolor': BG, 'savefig.facecolor': BG,
    'axes.edgecolor': FG, 'axes.linewidth': 0.5, 'axes.labelcolor': FG,
    'axes.titlecolor': FG, 'xtick.color': FG, 'ytick.color': FG, 'text.color': FG,
    'font.family': 'sans-serif', 'font.sans-serif': ['Helvetica Neue', 'Helvetica', 'DejaVu Sans'],
    'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': GRID, 'grid.linewidth': 0.5, 'grid.alpha': 0.6,
})

def is_gen(g, p):
    seen, x = set(), 1
    for _ in range(p - 1):
        x = (x * g) % p
        if x in seen: return False
        seen.add(x)
    return True
g_gen = next(g for g in range(2, P) if is_gen(g, P))
PM1 = P - 1

CKPT_PATH = hf_hub_download(
    repo_id='NikolayYudin/manifold-features-data',
    filename='four-algorithms/seed_1000_final.pt',
    repo_type='dataset',
)
print(f'Loaded checkpoint from: {CKPT_PATH}')

In [ ]:
# Cell 2 — architecture, load weights, compute per-class L2 manifold endpoints M[c]
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-8):
        super().__init__(); self.scale = nn.Parameter(torch.ones(d)); self.eps = eps
    def forward(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.scale

class GrokBlock(nn.Module):
    def __init__(self, d, nh, dropout=0.2):
        super().__init__()
        self.norm1 = RMSNorm(d); self.attn = nn.MultiheadAttention(d, nh, dropout=dropout, batch_first=True)
        self.norm2 = RMSNorm(d); self.w1 = nn.Linear(d, 4*d, bias=False)
        self.w2 = nn.Linear(4*d, d, bias=False); self.w3 = nn.Linear(d, 4*d, bias=False)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        h = self.norm1(x); o, _ = self.attn(h, h, h, need_weights=False)
        x = x + self.dropout(o)
        h2 = self.norm2(x); gate = F.silu(self.w1(h2))
        return x + self.dropout(self.w2(gate * self.w3(h2)))

class GrokModelShared(nn.Module):
    def __init__(self, vocab=VOCAB, d=384, nh=12, n_layers=2, out_p=P):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab, d); self.pos_emb = nn.Embedding(4, d)
        self.blocks = nn.ModuleList([GrokBlock(d, nh) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d)
        self.head = nn.Linear(d, out_p, bias=False)
    def to_L2(self, a, op_token, b):
        B = a.size(0)
        op = torch.full((B,), op_token, device=a.device, dtype=torch.long)
        eq = torch.full((B,), EQ, device=a.device, dtype=torch.long)
        tok = torch.stack([a, op, b, eq], dim=1)
        pos = torch.arange(4, device=a.device).unsqueeze(0).expand(B, -1)
        x = self.tok_emb(tok) + self.pos_emb(pos)
        for bl in self.blocks: x = bl(x)
        return self.norm_f(x)[:, -1, :]
    def forward(self, a, op_token, b):
        return self.head(self.to_L2(a, op_token, b))

m = GrokModelShared(d=384).to(device)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
m.load_state_dict(ckpt['state_dict']); m.eval()
print(f'Final accuracies in this checkpoint: {ckpt.get("final_accs", "n/a")}')

def make_task_data(task):
    aa, bb, cc = [], [], []
    for a in range(P):
        for b in range(P):
            if task == 'div' and b == 0: continue
            if task == 'div':    c = (a * pow(b, P-2, P)) % P
            elif task == 'add':  c = (a + b) % P
            elif task == 'max':  c = max(a, b)
            elif task == 'parity': c = (a + b) % 2
            aa.append(a); bb.append(b); cc.append(c)
    return (torch.tensor(aa, device=device), torch.tensor(bb, device=device),
            torch.tensor(cc, device=device))

@torch.no_grad()
def baseline_acc(task, batch=4096):
    a, b, c = make_task_data(task)
    correct = 0
    for i in range(0, len(a), batch):
        logits = m(a[i:i+batch], TASK_OP[task], b[i:i+batch])
        correct += (logits.argmax(-1) == c[i:i+batch]).sum().item()
    return correct / len(a)

print('\nBaseline accuracy (sanity):')
for t in TASKS: print(f'  {t}: {baseline_acc(t):.4f}')

@torch.no_grad()
def per_class_mean_L2(task, batch=4096):
    a, b, c = make_task_data(task)
    Hs = []
    for i in range(0, len(a), batch):
        Hs.append(m.to_L2(a[i:i+batch], TASK_OP[task], b[i:i+batch]).cpu().numpy())
    H = np.concatenate(Hs, axis=0); c_np = c.cpu().numpy()
    n_cls = 2 if task == 'parity' else P
    M_ = np.zeros((n_cls, H.shape[1]), dtype=np.float32)
    for cls in range(n_cls):
        idx = (c_np == cls)
        if idx.any(): M_[cls] = H[idx].mean(0)
    return M_

M = {t: per_class_mean_L2(t) for t in TASKS}
print('\nManifold endpoints computed:')
for t in TASKS: print(f'  M[{t}].shape = {M[t].shape}')

def step_fn(task, c, k):
    if task == 'add':    return (c + k) % P
    if task == 'div':
        if c == 0: return 0
        return (c * pow(g_gen, k % PM1, P)) % P
    if task == 'max':    return max(0, min(P-1, c + k))
    if task == 'parity': return (c + k) % 2

In [ ]:
# Cell 3 — three steering methods, swept over Delta in {0, +/-1, +/-2, +/-5, +/-10, +/-24, +/-50, +/-100}
@torch.no_grad()
def predict(h):
    return m.head(h).argmax(-1).cpu().numpy()

@torch.no_grad()
def walk(task, k, method='chord', n_samples=1000, seed=0):
    rng = np.random.RandomState(seed)
    a, b, c_gt = make_task_data(task)
    if n_samples < len(a):
        idx = rng.choice(len(a), n_samples, replace=False)
        a, b = a[idx], b[idx]
    h = m.to_L2(a, TASK_OP[task], b)
    c_pred = predict(h)
    target = np.array([step_fn(task, int(cp), k) for cp in c_pred])
    M_t = torch.from_numpy(M[task]).to(device)
    src = np.clip(c_pred, 0, M_t.shape[0]-1)
    tgt = np.clip(target, 0, M_t.shape[0]-1)
    if method == 'chord':
        delta = M_t[tgt] - M_t[src]
    elif method == 'linear':
        next1 = np.array([step_fn(task, int(cp), 1) for cp in c_pred])
        nx1 = np.clip(next1, 0, M_t.shape[0]-1)
        tangent = M_t[nx1] - M_t[src]
        delta = k * tangent
    elif method == 'random':
        chord = M_t[tgt] - M_t[src]
        rd = torch.randn_like(chord)
        delta = rd / (rd.norm(dim=-1, keepdim=True) + 1e-9) * chord.norm(dim=-1, keepdim=True)
    h_walked = h + delta
    return (predict(h_walked) == target).mean()

K_RANGE = [-100, -50, -24, -10, -5, -2, -1, 0, 1, 2, 5, 10, 24, 50, 100]
walk_results = {method: {t: {} for t in TASKS} for method in ['chord', 'linear', 'random']}
for method in ['chord', 'linear', 'random']:
    print(f'\n=== {method} ===')
    for t in TASKS:
        for k in K_RANGE:
            walk_results[method][t][k] = float(walk(t, k, method=method, n_samples=1000))
        row = ' '.join(f'k={k:+4d}:{walk_results[method][t][k]:.3f}' for k in K_RANGE)
        print(f'  {t}: {row}')

with open(OUT_DIR / 'walk_results.json', 'w') as f:
    json.dump({method: {t: {str(k): v for k, v in walk_results[method][t].items()} for t in TASKS}
               for method in ['chord', 'linear', 'random']}, f, indent=2)
print(f'\nSaved walk_results.json')

In [ ]:
# Cell 4 — full dictation: replace L2 residual with M_t[target] for every class
@torch.no_grad()
def dictate(task, target_c, n_samples=500, seed=0):
    rng = np.random.RandomState(seed)
    a, b, _ = make_task_data(task)
    n = min(n_samples, len(a))
    idx = rng.choice(len(a), n, replace=False)
    M_t = torch.from_numpy(M[task]).to(device)
    tgt = max(0, min(M_t.shape[0]-1, target_c))
    h_dict = M_t[tgt].unsqueeze(0).expand(n, -1)
    return (predict(h_dict) == tgt).mean()

dictation_results = {t: {} for t in TASKS}
for t in TASKS:
    n_cls = M[t].shape[0]
    hits = [dictate(t, c) for c in range(n_cls)]
    for c, h in zip(range(n_cls), hits): dictation_results[t][c] = float(h)
    arr = np.array(hits)
    print(f'  {t}: {n_cls} classes — mean={arr.mean():.4f}, frac@1.0={(arr==1.0).mean():.3f}')

with open(OUT_DIR / 'dictation_results.json', 'w') as f:
    json.dump({t: {str(c): v for c, v in dictation_results[t].items()} for t in TASKS}, f, indent=2)
print(f'\nSaved dictation_results.json')

In [ ]:
# Cell 5 — Figure 1: chord vs linear vs random hit rate per task
fig, axes = plt.subplots(1, 4, figsize=(15, 4), dpi=120, sharey=True)
for ax, t in zip(axes, TASKS):
    x = np.arange(len(K_RANGE))
    chord = [walk_results['chord'][t][k] for k in K_RANGE]
    linear = [walk_results['linear'][t][k] for k in K_RANGE]
    random = [walk_results['random'][t][k] for k in K_RANGE]
    ax.plot(x, chord, '-o', color=TASK_COLORS[t], linewidth=2.0, markersize=6, label='chord', zorder=3)
    ax.plot(x, linear, '--s', color=MUTED, linewidth=1.0, markersize=4, label='linear', zorder=2)
    ax.plot(x, random, ':^', color=GRID, linewidth=0.7, markersize=4, label='random', zorder=1)
    ax.axhline(1/P, color='gray', linestyle=':', linewidth=0.5, alpha=0.5)
    ax.set_xticks(x); ax.set_xticklabels([str(k) for k in K_RANGE], rotation=45, fontsize=7)
    ax.set_xlabel('step size k', fontsize=9); ax.set_title(t, fontsize=10)
    ax.legend(fontsize=7, loc='lower center', frameon=False); ax.set_ylim(-0.05, 1.05)
axes[0].set_ylabel('hit rate')
plt.suptitle('Manifold walk vs linear tangent vs random direction', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# Cell 6 — Figure 2: dictation hit rate per target class
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6), dpi=120, sharey=True)
for ax, t in zip(axes, TASKS):
    n_cls = M[t].shape[0]
    targets = list(range(n_cls))
    hits = [dictation_results[t][c] for c in targets]
    ax.bar(targets, hits, color=TASK_COLORS[t], alpha=0.85, width=1.0 if n_cls > 10 else 0.7)
    ax.axhline(1.0, color=FG, linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('target class', fontsize=9)
    ax.set_title(f'{t}  (mean={np.mean(hits):.4f})', fontsize=9)
    ax.set_ylim(-0.05, 1.08)
axes[0].set_ylabel('dictation hit rate')
plt.suptitle('Dictation: replacing the L2 residual with the manifold endpoint M[target]', fontsize=11)
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()

## Expected results

- **Chord walk**: 97-100% on all four tasks across the full Δ ∈ ±100 range.
- **Linear tangent**: fails almost immediately for the non-trivial tasks. A single tangent flies off the algorithm curve.
- **Random direction**: hits at chance ~1/P.
- **Dictation**: 100% on every target class for all four tasks.

If your numbers match within a percent or two, the geometry replicated.